In [ ]:
##############################################################################
# OPTIMIZATION & AI - LAB 07 - 2025/2026
##############################################################################

In [3]:
# System imports
import matplotlib.pyplot as plt
import numpy as np
import time

# Extra library imports
import pandas as pd
from scipy.optimize import linprog
from itertools import product

In [4]:
# User inputs
csvfile = 'projects.csv'
csvfile2 = 'projects-v2.csv'
max_budget = 1000000

More and more processes allow citizens to vote for public utility projects, such as participatory budgeting (see for example https://decider.paris.fr). In this lab, we will consider that each citizen can select only one project, and that the organizers select multiple projects according to the received votes, respecting usually a maximum budget to spend. This is in fact a constrained optimization problem! We will solve this problem combining linear programming and Branch and Bound.

# I.  Participatory budgeting

## a) Data exploration

Suppose we have a city allowing its citizens to vote for its favourite project among 5 candidates. The city cannot spend more than 1.000.000 euros to fund the selected projects. Let's have a look at the data!

<font color='blue'> Question 1: open the 'projects.csv' using pandas. Assign a variable "nb_projects" with the number of projects, for example using the *shape* attribute from Pandas. Display the available information for the 5 first projects using the *head* attribute. </font>

In [ ]:
# Load the first csv file
csvfile = 'projects.csv'
df = pd.read_csv(csvfile) 

# Get the number of projects
nb_projects = df.shape[0]

# Display the 5 first lines
df.head(5) 

FileNotFoundError: [Errno 2] No such file or directory: 'projects.csv'

We read for example that the cost of the first project is 261400 euros, and that it received 261 votes.

<font color='blue'> Question 2: supposing that each voter could choose only one project, how many people did vote? If all the projects were accepted, what would be the total cost? You may use the *sum* attribute from Pandas to answer these questions.</font> 

In [6]:
number_total_votes = df['Nb_votes'].sum()
print(number_total_votes)

number_total_cost = df['Cost'].sum()
print(number_total_cost)

3274
1705261


Supposing that the city cannot spend more than 1000000 euros (the "max_budget" variable) to fund these projects, you should conclude that unfortunately all the projects cannot be selected. Thus, we have to design a selection process. We choose to **select projects so that the total number of satisfied voters is maximized, while respecting the maximum budget constraint**.

We first solve this optimization problem using the brute force approach, namely computing all the possible and feasible solutions and extracting the best one.

## b) Project selection using brute force

Our problem can be formulated as follows:

$\displaystyle \max_{\boldsymbol x} \displaystyle \sum_{i=1}^n x_i v_i$ s.t. $\displaystyle \sum_{i=1}^n x_i c_i \leq \textrm{max_budget}$

with $n$ the number of available projects, $x_i$ the decision of selecting or not project $i$, $v_i$ the number of voters choosing project $i$, and $c_i$ the cost of project $i$.
Thus, $\boldsymbol x=[x_1, ..., x_n]$ is the vector of variables to optimize, each one being associated with a project. The expected solution is binary: $x_i = 1$ if project i is selected, $x_i = 0$ otherwise.

<font color='blue'> Question 3: what is the total number of possible solutions, without considering the maximum budget constraint? Write the answer as a function of the number of projects $n$, and display the numerical value. </font> 

<font color='green'> To complete </font> 

In [ ]:
# ... To complete ... 

<font color='blue'> Question 4: complete the following function to apply brute force on our constrained problem, *i.e* computing  all the possible solutions respecting the constraints. You may use the *product* function of the *itertool* library.</font> 

In [ ]:
def my_brute_force(v, c, max_budget):
    """
    Apply brute force on the linear maximization problem 
    
    Inputs
    ----------
    v: vector of integers
        the vote vector v = [v1, .., vn]
    c: vector of integers
       the cost vector c = [c1, .., cn]
    max_budget: integer
        the maximal budget 
        
    Outputs
    -------
    best_x: binary vector
        the vector of optimal variables
    best_fun: integer
        the optimal objective function value
        
    Command:
    best_x, best_fun = my_brute_force(v, c, b)
    
    """
    
    # Get the number of variables
    n = len(c)
    
    # Compute the best solution
    best_x = # ... To complete ...
    best_fun = # ... To complete ...
        
    return best_x, best_fun

<font color='blue'> Question 5: apply the Brute Force function to solve our maximization problem. Display the names of the selected projects, the total cost to spend, and the number of voters satisfied by the selected projects. <font color='blue'>

In [ ]:
# ... To complete ...

We just found the optimal solution ! However, brute force starts being too time consuming in a problem with more than 18 projects. To be able to generalize, we will now implement other approaches.

## c) Project selection using linear programming

We first investigate the linear programming approach. The selection process can then still be formulated as:

$\max \displaystyle \sum_{i=1}^n x_i v_i$ s.t. $\displaystyle \sum_{i=1}^n x_i c_i \leq \textrm{max_budget}$

The only difference is that in classical linear programming, the variables to optimize are not binary. 
To deal with this issue, we constrain $\boldsymbol x$ to represent the percentage of chance for a project to be selected. Thus, we force each variable $x_i$ to belong in $[0, 1]$.
We will now solve this linear programming problem using Scipy.

<font color='blue'> Question 6: complete the code below to solve this problem using linear programming.</font> 

In [ ]:
# Set the coefficients of the linear objective function to MINIMIZE
l = # ... To complete ...

# Set the inequality constraint matrix
A = # ... To complete ...

# Set the inequality constraint vector
b = # ... To complete ...

#Set the bounds
bounds = # ... To complete ...

# Perform the optimization using linear programming
res0 = linprog(l, A_ub=A, b_ub=b, bounds=bounds)

# Display the optimization result
print(res0)

# Save the optimal x vector in a variable
x0 = # ... To complete ...
print('\n x0 = {0}'.format(x0))

You should see that the optimal x vector is not binary: unfortunately, we cannot conclude directly. We first try a naive approach:  thresholding the output x vector.

<font color='blue'> Question 7: threshold each output variable at 0.5 (set $x_i$ to 1 if $x_i \geq 0.5$, 0 otherwise). Do you obtain a feasible solution ? Is it the optimal solution as expected in question 5?</font> 

In [ ]:
# ... To complete ...

<font color='green'> To complete </font> 

Thus, to properly solve our optimization problem, we cannot use linear programming alone... but we can combine it with the Branch and Bound method! 
That is in fact the basic way of solving binary integer programming problems, *i.e* linear programming problems with binary variables. 

## d) Project selection using Branch and Bound and linear programming

As we saw in course, Branch and Bound is a method enumerating in a clever way potential solutions, while progressively ignoring solutions that do not improve the current optimal solution. We present here a strategy to combine it with linear programming in a **maximization problem**. 

Let $\boldsymbol x=[x_1, ..., x_n]^T$ the vector of variables to optimize, and f the function to maximize.

Step 0. Initialization: apply linear programming constraining the variables to be in [0, 1] (this is called the relaxed problem). If the solution is binary, you're done.

Otherwise, repeat the 4 following steps until there is no child node to process:

Step 1. **Select $x_j$, the non-binary variable with the greatest value**: $ x_j = argmax \, x_i$ with $i \in \{1, n \}, x_i \notin \{0, 1\}$.

Step 2. Create two child nodes, $N_k$ and $N_{k+1}$, corresponding respectively to $x_j=0$ and $x_j=1$. Solve each associated linear programming relaxed problem (adding the constraints previously applied in the branch if existing). You obtain two optimal function values: $f^*(N_k)$ and $f^*(N_{k+1})$.

Step 3. For each child node: 
- If you obtain a binary solution, remove this node for further process (you can consider this step as a pruning): you cannot find a better binary solution on this branch. If the associated function value is greater than the current optimal function value (associated with an integer solution), update the current optimal function value.
- If you obtain no solution (infeasible problem), prune the associated node: all its potential child nodes will lead to infeasible problems.
- If you do not obtain a binary solution, set the **Upper Bound (UB)** of the current child node to the corresponding function value ($f^*(N_k)$ or $f^*(N_{k+1})$).

Step 4. If the maximal function value associated with a binary solution is greater than or equal to the maximal upper bound value associated with a childless node, stop the process. Otherwise, branch on the node with the **greatest Upper Bound**.

This process is illustrated in **BB-example.pdf** on a simpler example.

We will now implement this process for our maximization problem.

<font color='blue'> Question 8: complete the following function to solve the selection problem using both linear programming and Branch and Bound. </font> 

In [ ]:
def my_bb(x0, l, A, b):

    """
    Apply both Branch and Bound and Linear Programming
    on the linear maximization problem 
    
    Inputs
    ----------
    x0:binary vector
        the vector of initial variable values x0 = [x1, ..., xn]
    l: 1-D array
       the coefficients of the linear objective function to be minimized
    A: 2-D array 
       the inequality constraint matrix
    b: 1-D array
      the inequality constraint vector
        
    Outputs
    -------
    best_x: binary vector
        the vector of optimal variables
    best_fun: integer
        the optimal objective function value
        
    Command:
    best_x, best_fun = my_bb(x0, l, A, b)
    
    """
        
    # Initialize the node counter 
    node_cnt = 0

    # Get the number of projects
    nb_projects = len(x0)
    
    # Initialize the parent node of the current node
    parent_node = 0

    # Initialize the dictionary containing optimization
    # result for each node
    nodes_dict = {}

    # Initialize the stopping condition variable 
    to_process = True

    # Initialize optimization parameters
    best_fun = -np.inf
    best_node = None
    x = x0

    # Select the position in the x vector of the first variable to branch on, 
    # i.e the xi variable with the greatest NON BINARY value
    # in the initial x vector
    x_loc = # ... To complete ...

    while to_process:
        # Create the two child nodes (binary decision)
        for j in [0, 1]:

            # Update the current node counter
            node_cnt += 1

            # Set the bounds: xi = 0 or xi = 1 if xi already or currently
            # encountered in the branch, xi in [0, 1] otherwise
            if node_cnt <= 2:
                # Initialize the list of positions in the x vector of 
                # the variables already assigned to 0 or 1 at the current node
                node_fixed_x_locs = []

                # Initialize the list of values (0 or 1) of the variables 
                # already assigned at the current node
                node_fixed_x_vals = []
            else:
                # Get the sequence of decisions from the parent node
                node_fixed_x_locs = nodes_dict[parent_node]["fixed_x_loc"].copy()
                node_fixed_x_vals = nodes_dict[parent_node]["fixed_x_val"].copy()

            # Add the x position and value for the current node
            node_fixed_x_locs.append(# ... To complete ...)   
            node_fixed_x_vals.append(# ... To complete ...) 

            # Set the bounds
            bounds = # ... To complete ...

            # Perform the optimization using linear programming
            resi = linprog(l, A_ub=A, b_ub=b, bounds=bounds)

            # Success variable, taking value True if the optimization problem 
            # was feasible, False otherwise
            success = # ... To complete ...

            # Stop here on the branch if the problem is infeasible
            if success == False:
                continue
                
            # Optimal function value of the current MAXIMIZATION problem
            fun = # ... To complete ...

            # Optimal current vector of variables to optimize
            x = # ... To complete ...resi.x

            # Binary solution variable, taking value True if the current  
            # solution in binary, False otherwise
            binary_solution = # ... To complete ...

            # Do not go further if the optimal function value (associated with 
            # an integer solution) is greater than or equal to the maximal
            # upper bound value
            if # ... To complete ...:
                best_fun = fun
                best_x = x
                best_node = node_cnt

            # Add the node in the dictionary, but only if it is not pruned
            if # ... To complete ...:
                nodes_dict[node_cnt] = {"UB":fun,
                                       "x":x, 
                                       "fixed_x_loc":node_fixed_x_locs, 
                                       "fixed_x_val":node_fixed_x_vals}

        # Prune the parent node: remove it from the node dictionary 
        if node_cnt > 2:
            # ... To complete ...

        # Do not go further if there is no child node to create anymore
        if len(nodes_dict) == 0:
            to_process = False
        else:    
            # Compute the vector of upper bound values and node label
            ubs = np.array([node_dict["UB"] for node_dict in nodes_dict.values()])
            nodes = np.array(list(nodes_dict.keys()))
        
            # Do not go further if the optimal function value (associated with 
            # an integer solution) is greater than or equal to the maximal
            # remaining upper bound value
            if # ... To complete ...
                to_process = False
            else:      
                # Set the new node label to process as the one with the greatest
                # upper bound value
                parent_node = # ... To complete ...

                # Select the position in the x vector of the non-binary variable
                # with the greatest output xi value
                x = nodes_dict[parent_node]["x"]
                x_loc = # ... To complete ...

    return best_fun, best_x, node_cnt

<font color='blue'> Question 9: apply Branch and Bound on our problem. Compare the results and performances between this approach and the brute Force method. What approach should you favour on this dataset? </font> 

In [ ]:
# Perform B&B 
start = time.time()
best_fun, best_x, node_cnt = my_bb(x0, l, A, b)
end = time.time()

# ... To complete ...

<font color='green'> To complete </font> 

<font color='blue'> Question 10: repeat questions 5 and 9 on the file "projects-v2.csv". </font> 

In [ ]:
# ... To complete ...

<font color='green'> To complete </font> 